In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

print("Scraping courses ikt100 to ikt600")

BASE_URL = "https://www.uia.no/english/studies/courses/2026/spring/ikt{}.html"

headers = {
    "User-Agent": "Mozilla/5.0"
}

courses = []

def clean_text(tag):
    if not tag:
        return None
    return " ".join(tag.get_text(" ", strip=True).split())

# sections to skip
unwanted_words = ["contact", "resources", "about"]

# loop through course codes (adjust range if needed)
for i in range(100, 600):   # e.g. IKT100 → IKT599
    code = f"{i:03d}"
    url = BASE_URL.format(code)

    print(f"Trying {url}")

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 404:
            print("→ Not found, skipping")
            continue

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # if no title → not a valid course page
        h1 = soup.find("h1")
        if not h1:
            print("→ No title, skipping")
            continue

        course_data = {"url": url}
        course_data["title"] = clean_text(h1)

        # -------------------------
        # FACT BOX
        # -------------------------
        for dt in soup.find_all("dt"):
            key = clean_text(dt)
            dd = dt.find_next_sibling("dd")
            value = clean_text(dd)

            if key and value:
                normalized_key = (
                    key.lower()
                    .replace(":", "")
                    .replace(" ", "_")
                    .replace("-", "_")
                )

                if any(word in normalized_key for word in unwanted_words):
                    continue

                course_data[normalized_key] = value

        # -------------------------
        # H2 SECTIONS
        # -------------------------
        for h2 in soup.find_all("h2"):
            section_name = clean_text(h2)
            if not section_name:
                continue

            normalized_section = (
                section_name.lower()
                .replace(":", "")
                .replace(" ", "_")
                .replace("-", "_")
            )

            if any(word in normalized_section for word in unwanted_words):
                continue

            section_parts = []
            for sib in h2.find_next_siblings():
                if sib.name == "h2":
                    break

                if sib.name in ["p", "ul", "ol", "div"]:
                    text = sib.get_text(" ", strip=True)
                    if text:
                        section_parts.append(text)

            if section_parts:
                course_data[normalized_section] = "\n".join(section_parts)

        courses.append(course_data)
        print("→ Added")

    except requests.exceptions.RequestException as e:
        print(f"→ Error: {e}")
        continue

    # -------------------------
    # DELAY (IMPORTANT 🚨)
    # -------------------------
    sleep_time = random.uniform(1.5, 3.5)  # polite delay
    print(f"Sleeping {sleep_time:.2f}s\n")
    time.sleep(sleep_time)

# save
df = pd.DataFrame(courses)
df.to_csv("uia_all_ikt_courses.csv", index=False, encoding="utf-8-sig")

print(f"\nDone. Collected {len(df)} courses.")

Trying https://www.uia.no/english/studies/courses/2026/spring/ikt100.html
→ Not found, skipping
Trying https://www.uia.no/english/studies/courses/2026/spring/ikt101.html
→ Not found, skipping
Trying https://www.uia.no/english/studies/courses/2026/spring/ikt102.html
→ Not found, skipping
Trying https://www.uia.no/english/studies/courses/2026/spring/ikt103.html
→ Added
Sleeping 2.26s

Trying https://www.uia.no/english/studies/courses/2026/spring/ikt104.html
→ Added
Sleeping 1.83s

Trying https://www.uia.no/english/studies/courses/2026/spring/ikt105.html
→ Added
Sleeping 2.64s

Trying https://www.uia.no/english/studies/courses/2026/spring/ikt106.html
→ Not found, skipping
Trying https://www.uia.no/english/studies/courses/2026/spring/ikt107.html
→ Not found, skipping
Trying https://www.uia.no/english/studies/courses/2026/spring/ikt108.html
→ Not found, skipping
Trying https://www.uia.no/english/studies/courses/2026/spring/ikt109.html
→ Not found, skipping
Trying https://www.uia.no/english/

In [3]:
import os
import shutil
import pandas as pd
from tqdm.notebook import tqdm

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# -----------------------------
# 1) Load scraped data
# -----------------------------
df = pd.read_csv("uia_ikt_courses.csv").fillna("")

# -----------------------------
# 2) Helpers
# -----------------------------
def safe_str(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

section_fields = [
    "learning_outcomes",
    "contents",
    "teaching_and_learning_methods",
    "examinations",
    "recommended_previous_knowledge",
    "examination_requirements",
    "prerequisites",
    "other_information",
    "student_evaluation",
    "reduction_of_credits",
    "the_course_is_connected_to_the_following_study_programs",
    "offered_as_a_free_standing_course",
    "admission_requirement_if_given_as_a_free_standing_course",
]

documents = []

# Split only the section body, not the header
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# -----------------------------
# 3) Create documents
# -----------------------------
for i, row in tqdm(df.iterrows(), total=len(df), desc="Creating course documents"):
    base_meta = {
        "row_id": i,
        "url": safe_str(row.get("url", "")),
        "title": safe_str(row.get("title", "")),
        "ects_credits": safe_str(row.get("ects_credits", "")),
        "responsible_department": safe_str(row.get("responsible_department", "")),
        "course_leader": safe_str(row.get("course_leader", "")),
        "course_leaders": safe_str(row.get("course_leaders", "")),
        "lecture_semester": safe_str(row.get("lecture_semester", "")),
        "teaching_language": safe_str(row.get("teaching_language", "")),
        "duration": safe_str(row.get("duration", "")),
    }

    # Summary document
    summary_content = f"""
Course Title: {base_meta['title']}
URL: {base_meta['url']}
ECTS Credits: {base_meta['ects_credits']}
Responsible Department: {base_meta['responsible_department']}
Course Leader: {base_meta['course_leader']}
Course Leaders: {base_meta['course_leaders']}
Lecture Semester: {base_meta['lecture_semester']}
Teaching Language: {base_meta['teaching_language']}
Duration: {base_meta['duration']}
""".strip()

    documents.append(
        Document(
            page_content=summary_content,
            metadata={**base_meta, "section": "summary", "chunk_id": 0}
        )
    )

    # Section documents: split the VALUE only, then prepend header to each chunk
    for field in section_fields:
        value = safe_str(row.get(field, ""))
        if not value:
            continue

        section_title = field.replace("_", " ").title()

        header = f"""
Course Title: {base_meta['title']}
URL: {base_meta['url']}
Section: {section_title}
""".strip()

        # Split only the actual body text
        split_texts = text_splitter.split_text(value)

        for chunk_id, chunk_text in enumerate(split_texts):
            content = f"{header}\n\n{chunk_text}".strip()

            documents.append(
                Document(
                    page_content=content,
                    metadata={
                        **base_meta,
                        "section": field,
                        "chunk_id": chunk_id,
                    }
                )
            )

print(f"Final documents/chunks created directly: {len(documents)}")

# Optional preview
if documents:
    print("\nFirst document preview:\n")
    print(documents[0].page_content[:1000])
    print("\nMetadata:", documents[0].metadata)

# -----------------------------
# 4) Embedding model
# -----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# -----------------------------
# 5) Rebuild Chroma cleanly
# -----------------------------
persist_dir = "./chroma_langchain_db"

if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)

vectorstore = Chroma(
    collection_name="uia_courses",
    embedding_function=embeddings,
    persist_directory=persist_dir,
)

# -----------------------------
# 6) Index in batches
# -----------------------------
batch_size = 32

for start_idx in tqdm(range(0, len(documents), batch_size), desc="Indexing chunks"):
    batch = documents[start_idx:start_idx + batch_size]
    vectorstore.add_documents(batch)

print("Chunking and indexing complete.")

Creating course documents:   0%|          | 0/27 [00:00<?, ?it/s]

Final documents/chunks created directly: 280

First document preview:

Course Title: IKT103 Advanced Software Development (Spring 2026)
URL: https://www.uia.no/english/studies/courses/2026/spring/ikt103.html
ECTS Credits: 5.0
Responsible Department: Faculty of Engineering and Science
Course Leader: Christian Auby
Course Leaders: 
Lecture Semester: Spring
Teaching Language: Norwegian
Duration: ½ year

Metadata: {'row_id': 0, 'url': 'https://www.uia.no/english/studies/courses/2026/spring/ikt103.html', 'title': 'IKT103 Advanced Software Development (Spring 2026)', 'ects_credits': '5.0', 'responsible_department': 'Faculty of Engineering and Science', 'course_leader': 'Christian Auby', 'course_leaders': '', 'lecture_semester': 'Spring', 'teaching_language': 'Norwegian', 'duration': '½ year', 'section': 'summary', 'chunk_id': 0}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexing chunks:   0%|          | 0/9 [00:00<?, ?it/s]

InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database